## Imports

In [1]:
import pandas as pd
from surprise import Dataset, Reader, SVD, SVDpp
from surprise.model_selection import train_test_split
from surprise import accuracy
from sklearn.metrics.pairwise import cosine_similarity
from surprise.model_selection import GridSearchCV
from scipy.sparse import csr_matrix
import numpy as np
from sklearn.metrics import ndcg_score
import joblib
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import MultiLabelBinarizer



## Load & Preprocess Data

In [2]:

# Load datasets
ratings = pd.read_csv("./MovieLensDataset/rating.csv")
movies = pd.read_csv("./MovieLensDataset/movie.csv")


In [3]:
movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [4]:
ratings = ratings.drop('timestamp', axis=1)
ratings.head()

,userId,movieId,rating
0,1,2,3.5
1,1,29,3.5
2,1,32,3.5
3,1,47,3.5
4,1,50,3.5


In [5]:
movies.isnull().sum()

movieId    0
title      0
genres     0
dtype: int64

In [6]:
ratings.isnull().sum()

userId     0
movieId    0
rating     0
dtype: int64

In [7]:
#movies.shape
ratings.shape

(20000263, 3)

In [ ]:
merged_df = pd.merge(ratings, movies, on="movieId")

In [9]:
merged_df.head()

,userId,movieId,rating,title,genres
0,1,2,3.5,Jumanji (1995),Adventure|Children|Fantasy
1,1,29,3.5,"City of Lost Children, The (Cité des enfants p...",Adventure|Drama|Fantasy|Mystery|Sci-Fi
2,1,32,3.5,Twelve Monkeys (a.k.a. 12 Monkeys) (1995),Mystery|Sci-Fi|Thriller
3,1,47,3.5,Seven (a.k.a. Se7en) (1995),Mystery|Thriller
4,1,50,3.5,"Usual Suspects, The (1995)",Crime|Mystery|Thriller


In [10]:
merged_df.to_csv("merged_movies_ratings.csv", index=False)


# CF

In [12]:
user_map = {u: i for i, u in enumerate(merged_df['userId'].unique())}
movie_map = {m: i for i, m in enumerate(merged_df['movieId'].unique())}
user_inv_map = {i: u for u, i in user_map.items()}
movie_inv_map = {i: m for m, i in movie_map.items()}

rows = merged_df['userId'].map(user_map)
cols = merged_df['movieId'].map(movie_map)
vals = merged_df['rating']

user_item_sparse = csr_matrix(
    (vals, (rows, cols)), 
    shape=(len(user_map), len(movie_map))
)

In [13]:
print("✅ Sparse matrix shape:", user_item_sparse.shape)


✅ Sparse matrix shape: (138493, 26744)


In [15]:
item_similarity = cosine_similarity(user_item_sparse.T, dense_output=False)

def recommend_movies_cf(user_id, N=10):
    if user_id not in user_map:
        return pd.DataFrame({"Error": ["User not found"]})
    
    user_idx = user_map[user_id]
    user_ratings = user_item_sparse[user_idx].toarray().ravel()
    rated_movies = np.where(user_ratings > 0)[0]
    
    scores = {}
    for m in rated_movies:
        sim_scores = item_similarity[m].toarray().ravel()
        for idx, score in enumerate(sim_scores):
            if idx not in rated_movies and score > 0:
                scores[idx] = scores.get(idx, 0) + score * user_ratings[m]
    
    top_movies_idx = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:N]
    top_movie_ids = [movie_inv_map[idx] for idx, _ in top_movies_idx]
    
    return merged_df[merged_df["movieId"].isin(top_movie_ids)][["movieId", "title", "genres"]].drop_duplicates()

# SVD

In [ ]:
reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(merge_df[['userId', 'movieId', 'rating']], reader)
trainset, testset = train_test_split(data, test_size=0.2)

svd = SVD()
svd.fit(trainset)

predictions = svd.test(testset)
print("RMSE:", accuracy.rmse(predictions))

# Recommendation function
def recommend_movies(user_id, N=10):
    if user_id not in merge_df['userId'].unique():
        return pd.DataFrame({"Error": ["User not found"]})
    
    movie_ids = merge_df["movieId"].unique()
    user_movies = merge_df[merge_df["userId"] == user_id]["movieId"].tolist()
    
    preds = [(mid, svd.predict(user_id, mid).est) for mid in movie_ids if mid not in user_movies]
    preds.sort(key=lambda x: x[1], reverse=True)
    top_movies = [mid for mid, _ in preds[:N]]
    
    return merge_df[merge_df["movieId"].isin(top_movies)][["movieId", "title", "genres"]].drop_duplicates()


RMSE: 0.7856
RMSE: 0.7856112474153825


In [ ]:
# Evaluation Metrics
def precision_recall_at_k(predictions, k=10, threshold=3.5):
    user_est_true = {}
    for uid, _, true_r, est, _ in predictions:
        user_est_true.setdefault(uid, []).append((est, true_r))

    precisions, recalls = [], []
    for uid, ratings in user_est_true.items():
        ratings.sort(key=lambda x: x[0], reverse=True)
        top_k = ratings[:k]
        n_rel = sum((true_r >= threshold) for (_, true_r) in ratings)
        n_rec_k = sum((est >= threshold) for (est, true_r) in top_k)
        n_rel_and_rec_k = sum(((true_r >= threshold) and (est >= threshold)) for (est, true_r) in top_k)

        precisions.append(n_rel_and_rec_k / n_rec_k if n_rec_k != 0 else 0)
        recalls.append(n_rel_and_rec_k / n_rel if n_rel != 0 else 0)

    return sum(precisions) / len(precisions), sum(recalls) / len(recalls)

prec, rec = precision_recall_at_k(predictions, k=10)
print("Precision@10:", prec)
print("Recall@10:", rec)

# NDCG Example
user_est_true = {}
for uid, _, true_r, est, _ in predictions:
    user_est_true.setdefault(uid, []).append((est, true_r))

sample_uid = list(user_est_true.keys())[0]
user_est_true[sample_uid].sort(key=lambda x: x[0], reverse=True)

y_true = [[true_r for (_, true_r) in user_est_true[sample_uid][:10]]]
y_score = [[est for (est, _) in user_est_true[sample_uid][:10]]]

print("NDCG@10:", ndcg_score(y_true, y_score))

Precision@10: 0.7801994677205595
Recall@10: 0.5686201454161917
NDCG@10: 0.9120532798518184


In [ ]:
# 7. Example Usage
# =========================
print("\nCF Recommendations for user 1:")
recommend_movies_cf(user_id=1, N=10)

print("\nSVD Recommendations for user 1:")
print(recommend_movies(1, 5))


CF Recommendations for user 1:
                                                  title  \
1184  Star Wars: Episode VI - Return of the Jedi (1983)   
1237                               Groundhog Day (1993)   
1242                          Back to the Future (1985)   
1528                   Men in Black (a.k.a. MIB) (1997)   
2486                                 Matrix, The (1999)   

                       genres  
1184  Action|Adventure|Sci-Fi  
1237   Comedy|Fantasy|Romance  
1242  Adventure|Comedy|Sci-Fi  
1528     Action|Comedy|Sci-Fi  
2486   Action|Sci-Fi|Thriller  

SVD Recommendations for user 1:
                                                 title           genres
15701                                  For Neda (2010)      Documentary
16985  Playing the Victim (Izobrazhaya zhertvu) (2006)           Comedy
19233                                   The War (2007)  Documentary|War
23860    Zero Motivation (Efes beyahasei enosh) (2014)     Comedy|Drama
26440                       

# Hybrid NCF

In [ ]:
merge_df['genres'] = merge_df['genres'].apply(lambda x: x.split('|') if isinstance(x, str) else [])


In [ ]:
user2idx = {u: i for i, u in enumerate(merge_df['userId'].unique())}
item2idx = {m: i for i, m in enumerate(merge_df['movieId'].unique())}
merge_df['user'] = merge_df['userId'].map(user2idx)
merge_df['item'] = merge_df['movieId'].map(item2idx)

0    [Adventure, Animation, Children, Comedy, Fantasy]
1                       [Adventure, Children, Fantasy]
2                                    [Comedy, Romance]
3                             [Comedy, Drama, Romance]
4                                             [Comedy]
Name: genres, dtype: object
<class 'list'>


In [ ]:
mlb = MultiLabelBinarizer()
genre_features = pd.DataFrame(
    mlb.fit_transform(merge_df['genres']),
    columns=mlb.classes_,
    index=merge_df.index
)

merge_df = pd.concat([merge_df, genre_features], axis=1)
genre_cols = mlb.classes_

In [ ]:
genre_cols

In [ ]:
class HybridDataset(Dataset):
    def __init__(self, df, genre_cols):
        self.users = torch.tensor(df['user'].values, dtype=torch.long)
        self.items = torch.tensor(df['item'].values, dtype=torch.long)
        self.ratings = torch.tensor(df['rating'].values, dtype=torch.float32)
        self.genres = torch.tensor(df[genre_cols].values, dtype=torch.float32)
    
    def __len__(self):
        return len(self.ratings)
    
    def __getitem__(self, idx):
        return self.users[idx], self.items[idx], self.genres[idx], self.ratings[idx]


,(no genres listed),Action,Adventure,Animation,Children,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
movieId,,,,,,,,,,,,,,,,,,,,
1,0,0,1,1,1,1,0,0,0,1,0,0,0,0,0,0,0,0,0,0
2,0,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0
4,0,0,0,0,0,1,0,0,1,0,0,0,0,0,0,1,0,0,0,0
5,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
131254,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
131256,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0
131258,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


In [ ]:
train_df, temp_df = train_test_split(merge_df, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

train_dataset = HybridDataset(train_df, genre_cols)
val_dataset   = HybridDataset(val_df, genre_cols)
test_dataset  = HybridDataset(test_df, genre_cols)

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=256, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=256, shuffle=False)

In [ ]:
class HybridNCF(nn.Module):
    def __init__(self, num_users, num_items, num_genres, embedding_dim=64, hidden_layers=[128,64,32]):
        super(HybridNCF, self).__init__()
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.item_embedding = nn.Embedding(num_items, embedding_dim)
        self.genre_embedding = nn.Linear(num_genres, embedding_dim)

        layers = []
        input_size = embedding_dim * 3
        for h in hidden_layers:
            layers.append(nn.Linear(input_size, h))
            layers.append(nn.ReLU())
            input_size = h
        layers.append(nn.Linear(input_size, 1))
        self.mlp = nn.Sequential(*layers)

    def forward(self, user, item, genres):
        u = self.user_embedding(user)
        i = self.item_embedding(item)
        g = self.genre_embedding(genres)
        x = torch.cat([u, i, g], dim=-1)
        return self.mlp(x).squeeze()

,userId,movieId,rating,user,item,(no genres listed),Action,Adventure,Animation,Children,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,1,2,3.5,0,0,0,0,1,0,1,...,0,0,0,0,0,0,0,0,0,0
1,1,29,3.5,0,1,0,0,1,0,0,...,0,0,0,0,1,0,1,0,0,0
2,1,32,3.5,0,2,0,0,0,0,0,...,0,0,0,0,1,0,1,1,0,0
3,1,47,3.5,0,3,0,0,0,0,0,...,0,0,0,0,1,0,0,1,0,0
4,1,50,3.5,0,4,0,0,0,0,0,...,0,0,0,0,1,0,0,1,0,0


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_users = len(user2idx)
num_items = len(item2idx)
num_genres = len(genre_cols)

model = HybridNCF(num_users, num_items, num_genres).to(device)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(10):
    model.train()
    total_loss = 0
    for user, item, genres, rating in train_loader:
        user, item, genres, rating = user.to(device), item.to(device), genres.to(device), rating.to(device)
        pred = model(user, item, genres)
        loss = criterion(pred, rating)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_train_loss = total_loss / len(train_loader)

    # Validation
    model.eval()
    val_loss = 0
    with torch.no_grad():
        for user, item, genres, rating in val_loader:
            user, item, genres, rating = user.to(device), item.to(device), genres.to(device), rating.to(device)
            pred = model(user, item, genres)
            loss = criterion(pred, rating)
            val_loss += loss.item()
    avg_val_loss = val_loss / len(val_loader)

    print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val Loss: {avg_val_loss:.4f}")


In [ ]:
model.eval()
predictions, test_ratings, test_users, test_items = [], [], [], []

with torch.no_grad():
    for user, item, genres, rating in test_loader:
        user, item, genres, rating = user.to(device), item.to(device), genres.to(device), rating.to(device)
        pred = model(user, item, genres)
        predictions.extend(pred.cpu().numpy())
        test_ratings.extend(rating.cpu().numpy())
        test_users.extend(user.cpu().numpy())
        test_items.extend(item.cpu().numpy())

predictions = np.array(predictions)
test_ratings = np.array(test_ratings)
rmse = np.sqrt(mean_squared_error(test_ratings, predictions))
print(f"Test RMSE: {rmse:.4f}")

In [ ]:
# 9. Precision, Recall, NDCG
def precision_recall_ndcg_at_k(users, items, true_ratings, pred_ratings, k=10, threshold=3.5):
    user_ratings = defaultdict(list)
    for u, i, t, p in zip(users, items, true_ratings, pred_ratings):
        user_ratings[u].append((p, t))
    
    precisions, recalls, ndcgs = [], [], []
    for u, ratings in user_ratings.items():
        ratings.sort(key=lambda x: x[0], reverse=True)
        top_k = ratings[:k]
        
        n_rel = sum((true_r >= threshold) for (_, true_r) in ratings)
        n_rel_and_rec_k = sum((true_r >= threshold) for (_, true_r) in top_k)
        
        precisions.append(n_rel_and_rec_k / k if k != 0 else 0)
        recalls.append(n_rel_and_rec_k / n_rel if n_rel != 0 else 0)
        
        dcg = sum((2**true_r - 1)/np.log2(idx+2) for idx, (_, true_r) in enumerate(top_k))
        idcg = sum((2**true_r - 1)/np.log2(idx+2) for idx, (_, true_r) in enumerate(sorted(ratings, key=lambda x:x[1], reverse=True)[:k]))
        ndcgs.append(dcg/idcg if idcg > 0 else 0)
        
    return np.mean(precisions), np.mean(recalls), np.mean(ndcgs)

precision, recall, ndcg = precision_recall_ndcg_at_k(
    test_users, test_items, test_ratings, predictions, k=10, threshold=3.5
)


Train size: 16000210
Validation size: 2000026
Test size: 2000027


In [ ]:
print(f"Precision@10: {precision:.4f}")
print(f"Recall@10: {recall:.4f}")
print(f"NDCG@10: {ndcg:.4f}")

In [ ]:

# Save model
joblib.dump(model, "HybridNCF.pkl")
print("The model is saved...")

NameError: name 'svd' is not defined